In [0]:
# Databricks environment configuration
catalog_name = "learning_projects"
schema_name = "retail_project"

table_prefix = f"{catalog_name}.{schema_name}"

In [0]:
silver_customers_df = spark.table(
    f"{table_prefix}.silver_customers"
)

silver_orders_df = spark.table(
    f"{table_prefix}.silver_orders"
)
print("customers :", silver_customers_df.count())
print("orders :", silver_orders_df.count())

In [0]:
from pyspark.sql.functions import col
customer_history_df = (
    silver_customers_df.alias("c")
    .join(silver_orders_df.alias("o"),
          col("o.customer_id") == col("c.customer_id"), "inner")
    .select(
        col("o.customer_id").alias("customer_id"),
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
        "order_purchase_timestamp"
    )
)

In [0]:
print("customer history rows: ", customer_history_df.count())
customer_history_df.show(10, truncate= False)

In [0]:
from pyspark.sql.functions import lag, col
from pyspark.sql.window import Window
window = Window.partitionBy("customer_unique_id").orderBy("order_purchase_timestamp")
customer_history_df= (
    customer_history_df
    .withColumn("previous_zip",
                lag("customer_zip_code_prefix").over(window))
    .withColumn("previous_city",
                lag("customer_city").over(window))
    .withColumn("previous_state",
                lag("customer_state").over(window))
)

In [0]:
from pyspark.sql.functions import count, col
(
    customer_history_df
    .groupBy("customer_unique_id")
    .agg(count("*").alias("count"))
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
    #.show(truncate=False)
)
(
    customer_history_df
    .filter(col("customer_unique_id")== "8d50f5eadf50201ccdcedfb9e2ac8455")
    .orderBy(col("order_purchase_timestamp").desc())
    .show(truncate = False)
)

In [0]:
from pyspark.sql.functions import col, when, row_number
from pyspark.sql.window import Window

customer_window = (
    Window.partitionBy("customer_unique_id")
    .orderBy("order_purchase_timestamp", "customer_id")
)

# Identify the first record for each customer
customer_history_df = customer_history_df.withColumn(
    "customer_row_num",
    row_number().over(customer_window)
)

customer_history_df = customer_history_df.withColumn(
    "is_address_change",
    when(
        (col("customer_row_num") == 1)
        | (~col("customer_zip_code_prefix").eqNullSafe(col("previous_zip")))
        | (~col("customer_city").eqNullSafe(col("previous_city")))
        | (~col("customer_state").eqNullSafe(col("previous_state"))),
        1
    ).otherwise(0)
)


In [0]:
customer_changes_df = customer_history_df.filter(
    col("is_address_change") == 1
)

print("Customer change records:", customer_changes_df.count())

In [0]:
from pyspark.sql.functions import lead

window = Window.partitionBy("customer_unique_id").orderBy("order_purchase_timestamp")
customer_changes_df = (
    customer_changes_df
    .withColumn("effective_from", col("order_purchase_timestamp"))
    .withColumn("effective_to",
                lead("order_purchase_timestamp").over(window))
    .withColumn("is_current",
                when (col("effective_to").isNull(), True).otherwise(False))
)
customer_changes_df.groupBy("is_current").count().show()


In [0]:
from pyspark.sql.functions import row_number, countDistinct, col

from pyspark.sql.window import Window

surrogate_key_window = Window.orderBy(
    "customer_unique_id",
    "effective_from"
)
dim_customer_df = (
    customer_changes_df
    .withColumn("customer_sk", row_number().over(surrogate_key_window))
    .select(
        "customer_sk",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
        "effective_from",
        "effective_to",
        "is_current"

    )
)
## Below is for data validation checkings
dim_customer_df.show(10)
print("total dimension rows:", dim_customer_df.count())
distinct_sk = (
    dim_customer_df
    .select("customer_sk")
    .distinct()
    .count()
)

print("Distinct customer_sk:", distinct_sk)
current_rows = (
    dim_customer_df
    .filter(col("is_current") == True)
    .count()
)

print("Current rows:", current_rows)

multiple_current_df = (
    dim_customer_df
    .filter(col("is_current") == True)
    .groupBy("customer_unique_id")
    .count()
    .filter(col("count") > 1)
)

print(
    "Customers with multiple current records:",
    multiple_current_df.count()
)

In [0]:
(
    dim_customer_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{table_prefix}.dim_customer"
    )
)

In [0]:
spark.sql(f"""
    SELECT *
    FROM {table_prefix}.dim_customer
    LIMIT 10
""").show(truncate=False)

In [0]:
silver_product_df = spark.table(f"{table_prefix}.silver_products")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.orderBy("product_id")
dim_product_df = (
    silver_product_df
    .withColumn("product_sk", row_number().over(window))
    .select(
        "product_sk",
        "product_id",
        "product_category_name",
        "product_name_length",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    )
)

# validate
print("total dimension rows: ", dim_product_df.count())
distinct_product_id = (
    dim_product_df
    .agg(countDistinct("product_id").alias("product_id_distinct"))
)
distinct_product_id.show()
distinct_product_sk = (
    dim_product_df
    .agg(countDistinct("product_sk").alias("product_sk_distinct"))
)
distinct_product_sk.show()

In [0]:
(
    dim_product_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{table_prefix}.dim_product"
    )
)

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT product_sk) AS distinct_product_sk,
    COUNT(DISTINCT product_id) AS distinct_product_id
FROM learning_projects.retail_project.dim_product;

In [0]:
from pyspark.sql.functions import col, countDistinct

customer_history_df.groupBy("is_address_change") \
    .count() \
    .orderBy("is_address_change") \
    .show()

customer_history_df.agg(
    countDistinct("customer_unique_id").alias("unique_customers")
).show()

customer_history_df.filter(
    col("is_address_change") == 1
).agg(
    countDistinct("customer_unique_id").alias("customers_with_change_records")
).show()

In [0]:
from pyspark.sql.functions import col, count

# Find customers with zero or multiple current records
current_record_issues_df = (
    dim_customer_df
    .groupBy("customer_unique_id")
    .agg(
        count("*").alias("total_versions"),
        count(
            when(col("is_current") == True, 1)
        ).alias("current_versions")
    )
    .filter(col("current_versions") != 1)
)

print("Total dimension rows:", dim_customer_df.count())
print("Customers with invalid current-record counts:",
      current_record_issues_df.count())

In [0]:
from pyspark.sql.functions import countDistinct

total_rows = dim_customer_df.count()

distinct_keys = (
    dim_customer_df
    .agg(countDistinct("customer_sk").alias("distinct_keys"))
    .first()["distinct_keys"]
)

print("Total dimension rows:", total_rows)
print("Distinct surrogate keys:", distinct_keys)
print("Surrogate keys unique:", total_rows == distinct_keys)

In [0]:
from pyspark.sql.functions import col

# Check historical records with invalid date ranges
invalid_date_ranges_df = dim_customer_df.filter(
    (col("effective_to").isNotNull()) &
    (col("effective_to") <= col("effective_from"))
)

# Check current records with incorrect end dates
invalid_current_dates_df = dim_customer_df.filter(
    (col("is_current") == True) &
    (col("effective_to").isNotNull())
)

# Check historical records missing end dates
invalid_historical_dates_df = dim_customer_df.filter(
    (col("is_current") == False) &
    (col("effective_to").isNull())
)

print("Invalid effective-date ranges:", invalid_date_ranges_df.count())
print("Current records with end dates:", invalid_current_dates_df.count())
print("Historical records missing end dates:", invalid_historical_dates_df.count())

In [0]:
from pyspark.sql.functions import countDistinct, col

total_products = dim_product_df.count()

distinct_product_ids = (
    dim_product_df
    .select("product_id")
    .distinct()
    .count()
)

distinct_product_sks = (
    dim_product_df
    .select("product_sk")
    .distinct()
    .count()
)

null_product_ids = dim_product_df.filter(
    col("product_id").isNull()
).count()

print("Total product dimension rows:", total_products)
print("Distinct product IDs:", distinct_product_ids)
print("Distinct product surrogate keys:", distinct_product_sks)
print("Null product IDs:", null_product_ids)
print("Product IDs unique:", total_products == distinct_product_ids)
print("Surrogate keys unique:", total_products == distinct_product_sks)